# Project status

This file will be showing the status of this project. I think that in terms of code and functionality this codebase is quite beautiful. However, in disease prediction it sucks! There is some disparity between how my models use the graph structure. They don't. 

In [ ]:
import importlib.util
import sys
from datetime import date
import pandas as pd
import numpy as np
from typing import Dict, Literal, Optional
import matplotlib.pyplot as plt
import seaborn as sns
import matplotlib.dates as mdates
import os
from pathlib import Path

wissdaten_dir    = os.environ.get('TMPDIR') +'/wissdaten/'
data_env         = os.path.join(wissdaten_dir, 'ZKI-PH4/deschrijvers_wissdaten/data')

data_module      = module_path = data_env + "/__init__.py"
spec = importlib.util.spec_from_file_location("data", module_path)
module = importlib.util.module_from_spec(spec)
sys.modules["data"] = module
spec.loader.exec_module(module)

from data.visuals import *
colors_dict = {}
colors_dict['context']    = hexcodes['sky blue']
colors_dict['future']     = hexcodes['soft red']
colors_dict['predictions']= hexcodes['moss green']

In [2]:
import torch
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from typing import Dict, List, Tuple, Optional
from torch_geometric.data import Data
import os
import sys

from tests import *

from src.dataloading.gnndataloader import GNNDataLoader
from src.dataloading.epidataloader import EpiDataLoader
from src.models.temporal_gcn import TemporalGCNModel
from src.models.spatial_gcn import SpatialGCNModel
from src.models.node_lstm import NodeLSTM

n_periods = 8
n_epochs  = 50
horizon   = 1
lags      = range(4,5)


epidata = EpiDataLoader('influenza', data_env, aggr_level= '03', min_date='2012-06-01',max_date='2020-06-01')
# epidata.add_time_features()
epidata.log_transform_target()
epidata.normalize('2018-06-01','2019-06-01','zscore')
epidata.add_lagged_features(lags = lags)

gnnloader1 = GNNDataLoader(epidata).retrieve_graph('identity_graph').construct_dataloaders(periods=n_periods, prediction_horizon=horizon)
gnnloader2 = GNNDataLoader(epidata).retrieve_graph('boolean_neighbors').construct_dataloaders(periods=n_periods, prediction_horizon=horizon)
gnnloader3 = GNNDataLoader(epidata).retrieve_graph('gravity_model_log').construct_dataloaders(periods=n_periods, prediction_horizon=horizon)
gnnloader4 = GNNDataLoader(epidata).retrieve_graph('gravity_model_top8_log').construct_dataloaders(periods=n_periods, prediction_horizon=horizon)

dataloaders = {'identity_graph': gnnloader1,
                    'boolean_neighbors': gnnloader2,
                    'gravity_model1': gnnloader3,
                    'gravity_model2': gnnloader4}

models = {}

model_classes    = {'tgcn': TemporalGCNModel,
                    'sgcn': SpatialGCNModel}

for modeltype, cl in model_classes.items():

    for dataloader_name, dataloader in dataloaders.items():

        graph_validation = GraphValidator(dataloader, graphname = dataloader_name)
        graph_validation.run_tests()

        name = modeltype + "_" + dataloader_name 

        ml = cl(name = name, dataloader = dataloader)
        ml.set_model_hparams()
        ml.set_global_hparams(lr = 0.00005, n_epochs=n_epochs, scheduler_kwargs={'step_size':15}, min_delta = 0.1)
        ml.train(show_loss = False)
        ml.forecast()
        # ml.show_forecasts(norm = True)

        models[name] = ml

        validator = GraphUsageValidator(ml)
        validator.run_tests()

/home/de-schrijvers/.conda/envs/gnenv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/home/de-schrijvers/.conda/envs/gnenv/lib/python3.11/site-packages/geopandas/geodataframe.py:1968: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  super().__setitem__(key, value)


  - gravity_model_dense_k25 (highest density)
  - k_nearest_dense_k30 (good connectivity)
  - distance_threshold_dense_300k (distance-based)

In [7]:
#!/usr/bin/env python3
"""
Test Improved Models with Dense Graphs

This script tests the improved GNN models with the newly created dense graphs
to verify better graph utilization and performance.
"""

import os
import sys
import torch
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

from src.dataloading import EpiDataLoader
from src.dataloading.gnndataloader import GNNDataLoader
from src.models.improved_temporal_gcn import ImprovedTemporalGCNModel
from src.models.improved_spatial_gcn import ImprovedSpatialGCNModel
from tests.graphusage_validator import GraphUsageValidator
from tests.graphvalidator import GraphValidator



def test_single_model(model_class, 
                      model_name, dataloader, graph_type, test_config):
    """Test a single model with given configuration."""
    print(f"\nTesting {model_name} with {graph_type}...")
    
    try:
        # Create model
        model = model_class(name=f'{model_name}_{graph_type}', dataloader=dataloader)
        
        # Set model hyperparameters
        if hasattr(model, 'set_model_hparams'):
            model.set_model_hparams(**test_config['model_hparams'])
        
        # Set global hyperparameters
        if hasattr(model, 'set_global_hparams'):
            model.set_global_hparams(**test_config['global_hparams'])
        
        # Train model
        print(f"  Training {model_name}...")
        model.train()
        
        # Forecast
        print(f"  Forecasting with {model_name}...")
        model.forecast()
        
        # Run validation tests
        print(f"  Running validation tests for {model_name}...")
        validator = GraphUsageValidator(model)
        validator.run_tests()
        
        return {
            'model': model,
            'validator': validator,
            'test_loss': getattr(model, 'test_loss', None),
            'success': True
        }
        
    except Exception as e:
        print(f"  Error testing {model_name}: {e}")
        return {
            'model': None,
            'validator': None,
            'test_loss': None,
            'success': False,
            'error': str(e)
        }


def test_graph_type(epidata, graph_type, test_configs):
    """Test all models with a specific graph type."""
    print(f"\n{'='*60}")
    print(f"TESTING GRAPH TYPE: {graph_type}")
    print(f"{'='*60}")
    
    try:
        # Load graph and create dataloader
        gnnloader = GNNDataLoader(epidata).retrieve_graph(graph_type).construct_dataloaders(periods=8)
        
        # Validate graph structure
        print(f"Validating graph structure for {graph_type}...")
        graph_validator = GraphValidator(gnnloader, graph_type)
        graph_validator.run_tests()
        
        print(f"Graph stats: {graph_validator.logs}")
        
        results = {
            'graph_type': graph_type,
            'graph_stats': graph_validator.logs,
            'models': {}
        }
        
        # Test each model configuration
        for config_name, config in test_configs.items():
            print(f"\nTesting configuration: {config_name}")
            
            # Test Improved Temporal GCN
            tgcn_result = test_single_model(
                ImprovedTemporalGCNModel, 
                'improved_tgcn', 
                gnnloader, 
                graph_type, 
                config
            )
            
            # Test Improved Spatial GCN
            sgcn_result = test_single_model(
                ImprovedSpatialGCNModel, 
                'improved_sgcn', 
                gnnloader, 
                graph_type, 
                config
            )
            
            results['models'][config_name] = {
                'tgcn': tgcn_result,
                'sgcn': sgcn_result
            }
        
        return results
        
    except Exception as e:
        print(f"Error testing graph type {graph_type}: {e}")
        return {
            'graph_type': graph_type,
            'error': str(e),
            'models': {}
        }


def run_comprehensive_tests(epidata):
    """Run comprehensive tests on multiple graph types and model configurations."""
    print("\n" + "="*60)
    print("COMPREHENSIVE MODEL TESTING")
    print("="*60)
    
    # Define test configurations
    test_configs = {
        'default': {
            'model_hparams': {
                'hidden_size': 64,
                'num_layers': 3,
                'temporal_layers': 2,
                'dropout': 0.3,
                'use_attention': True,
                'use_residual': True
            },
            'global_hparams': {
                'lr': 0.00005,
                'n_epochs': 30,
                'scheduler_kwargs': {'step_size': 10, 'gamma': 0.7},
                'min_delta': 0.001
            }
        },
        'high_capacity': {
            'model_hparams': {
                'hidden_size': 128,
                'num_layers': 4,
                'temporal_layers': 3,
                'dropout': 0.2,
                'use_attention': True,
                'use_residual': True
            },
            'global_hparams': {
                'lr': 0.00005,
                'n_epochs': 50,
                'scheduler_kwargs': {'step_size': 15, 'gamma': 0.8},
                'min_delta': 0.0005
            }
        }
    }
    
    # Define graph types to test
    graph_types = [
        'identity_graph',  # Baseline
        'boolean_neighbors',  # Original sparse
        'gravity_model_log',  # Original sparse
        'boolean_neighbors_dense',  # New dense
        'gravity_model_dense_k15',  # New dense
        'gravity_model_dense_k20',  # New dense
        'k_nearest_dense',  # New dense
        'distance_threshold_dense_200k',  # New dense
    ]
    
    all_results = {}
    
    for graph_type in graph_types:
        try:
            results = test_graph_type(epidata, graph_type, test_configs)
            all_results[graph_type] = results
        except Exception as e:
            print(f"Failed to test {graph_type}: {e}")
            all_results[graph_type] = {'error': str(e)}
    
    return all_results


def analyze_results(all_results):
    """Analyze and summarize test results."""
    print("\n" + "="*60)
    print("RESULTS ANALYSIS")
    print("="*60)
    
    # Create summary data
    summary_data = []
    
    for graph_type, result in all_results.items():
        if 'error' in result:
            print(f"Skipping {graph_type} due to error: {result['error']}")
            continue
        
        graph_stats = result['graph_stats']
        
        for config_name, config_results in result['models'].items():
            for model_type in ['tgcn', 'sgcn']:
                model_result = config_results[model_type]
                
                if model_result['success'] and model_result['validator']:
                    validation_logs = model_result['validator'].logs
                    
                    summary_data.append({
                        'Graph Type': graph_type,
                        'Config': config_name,
                        'Model': model_type.upper(),
                        'Density': graph_stats['density'],
                        'Edges': graph_stats['num_edges'],
                        'Using Graph': validation_logs['test1']['is_using_graph'],
                        'Cosine Similarity': validation_logs['test1']['cosine_similarity'],
                        'Edge Sensitive': validation_logs['test3']['is_edge_sensitive'],
                        'Test Loss': model_result['test_loss']
                    })
    
    if not summary_data:
        print("No successful test results to analyze!")
        return
    
    # Create summary dataframe
    summary_df = pd.DataFrame(summary_data)
    
    print("\nSUMMARY TABLE:")
    print(summary_df.to_string(index=False))
    
    # Save results
    summary_df.to_csv('data/graphs/model_test_results.csv', index=False)
    print(f"\nResults saved to: data/graphs/model_test_results.csv")
    
    return summary_df


def create_visualizations(summary_df):
    """Create visualizations of the test results."""
    print("\n" + "="*60)
    print("CREATING VISUALIZATIONS")
    print("="*60)
    
    if summary_df.empty:
        print("No data to visualize!")
        return
    
    # Create comprehensive visualization
    fig, axes = plt.subplots(2, 3, figsize=(18, 12))
    
    # Plot 1: Graph density vs cosine similarity
    for model in summary_df['Model'].unique():
        model_data = summary_df[summary_df['Model'] == model]
        axes[0,0].scatter(model_data['Density'], model_data['Cosine Similarity'], 
                         label=model, alpha=0.7, s=100)
    
    axes[0,0].set_xlabel('Graph Density')
    axes[0,0].set_ylabel('Cosine Similarity (Identity vs Real)')
    axes[0,0].set_title('Graph Density vs Cosine Similarity\n(Lower = Better Graph Utilization)')
    axes[0,0].legend()
    axes[0,0].grid(True, alpha=0.3)
    
    # Plot 2: Test loss comparison
    pivot_loss = summary_df.pivot_table(values='Test Loss', index='Graph Type', 
                                       columns='Model', aggfunc='mean')
    pivot_loss.plot(kind='bar', ax=axes[0,1], alpha=0.7)
    axes[0,1].set_xlabel('Graph Type')
    axes[0,1].set_ylabel('Test Loss')
    axes[0,1].set_title('Test Loss Comparison')
    axes[0,1].tick_params(axis='x', rotation=45)
    axes[0,1].legend()
    axes[0,1].grid(True, alpha=0.3)
    
    # Plot 3: Edge sensitivity
    pivot_sensitive = summary_df.pivot_table(values='Edge Sensitive', index='Graph Type', 
                                            columns='Model', aggfunc='mean')
    pivot_sensitive.plot(kind='bar', ax=axes[0,2], alpha=0.7)
    axes[0,2].set_xlabel('Graph Type')
    axes[0,2].set_ylabel('Edge Sensitive (1=Yes, 0=No)')
    axes[0,2].set_title('Edge Sensitivity\n(1 = Model responds to graph changes)')
    axes[0,2].tick_params(axis='x', rotation=45)
    axes[0,2].legend()
    axes[0,2].grid(True, alpha=0.3)
    
    # Plot 4: Graph utilization
    pivot_using = summary_df.pivot_table(values='Using Graph', index='Graph Type', 
                                        columns='Model', aggfunc='mean')
    pivot_using.plot(kind='bar', ax=axes[1,0], alpha=0.7)
    axes[1,0].set_xlabel('Graph Type')
    axes[1,0].set_ylabel('Using Graph (1=Yes, 0=No)')
    axes[1,0].set_title('Graph Utilization\n(1 = Model behaves differently with identity vs real graphs)')
    axes[1,0].tick_params(axis='x', rotation=45)
    axes[1,0].legend()
    axes[1,0].grid(True, alpha=0.3)
    
    # Plot 5: Graph density vs test loss
    for model in summary_df['Model'].unique():
        model_data = summary_df[summary_df['Model'] == model]
        axes[1,1].scatter(model_data['Density'], model_data['Test Loss'], 
                         label=model, alpha=0.7, s=100)
    
    axes[1,1].set_xlabel('Graph Density')
    axes[1,1].set_ylabel('Test Loss')
    axes[1,1].set_title('Graph Density vs Test Loss')
    axes[1,1].legend()
    axes[1,1].grid(True, alpha=0.3)
    
    # Plot 6: Summary heatmap
    heatmap_data = summary_df.pivot_table(values=['Using Graph', 'Edge Sensitive'], 
                                         index='Graph Type', columns='Model', aggfunc='mean')
    sns.heatmap(heatmap_data, annot=True, cmap='RdYlGn', center=0.5, ax=axes[1,2])
    axes[1,2].set_title('Graph Utilization Summary\n(Green = Good, Red = Poor)')
    
    plt.tight_layout()
    plt.savefig('data/graphs/model_test_results.png', dpi=300, bbox_inches='tight')
    plt.show()
    
    print("Visualization saved to: data/graphs/model_test_results.png")


def print_key_insights(summary_df):
    """Print key insights from the test results."""
    print("\n" + "="*60)
    print("KEY INSIGHTS")
    print("="*60)
    
    if summary_df.empty:
        print("No results to analyze!")
        return
    
    # Find best performing graphs
    best_using_graph = summary_df[summary_df['Using Graph'] == True]
    best_edge_sensitive = summary_df[summary_df['Edge Sensitive'] == True]
    lowest_cosine_sim = summary_df.loc[summary_df['Cosine Similarity'].idxmin()]
    
    print("1. GRAPH UTILIZATION:")
    if not best_using_graph.empty:
        print(f"   Models successfully using graph structure:")
        for _, row in best_using_graph.iterrows():
            print(f"   - {row['Model']} with {row['Graph Type']} (cosine sim: {row['Cosine Similarity']:.4f})")
    else:
        print("   ⚠️  No models are successfully using graph structure!")
    
    print("\n2. EDGE SENSITIVITY:")
    if not best_edge_sensitive.empty:
        print(f"   Models sensitive to edge changes:")
        for _, row in best_edge_sensitive.iterrows():
            print(f"   - {row['Model']} with {row['Graph Type']}")
    else:
        print("   ⚠️  No models are sensitive to edge changes!")
    
    print(f"\n3. BEST GRAPH UTILIZATION:")
    print(f"   {lowest_cosine_sim['Model']} with {lowest_cosine_sim['Graph Type']}")
    print(f"   Cosine similarity: {lowest_cosine_sim['Cosine Similarity']:.4f}")
    print(f"   Graph density: {lowest_cosine_sim['Density']:.4f}")
    
    print("\n4. RECOMMENDATIONS:")
    if best_using_graph.empty:
        print("   ⚠️  CRITICAL: Models are still not using graph structure!")
        print("   - Try even denser graphs")
        print("   - Increase model complexity")
        print("   - Adjust training parameters")
    else:
        print("   ✓ Models are using graph structure!")
        print("   - Use the best performing graph types for production")
        print("   - Consider the trade-off between density and performance")


def main():
    """Main function to test improved models."""
    print("IMPROVED MODEL TESTING SCRIPT")
    print("="*60)
    print("This script tests improved GNN models with dense graphs.")
    
    try:

        
        # Run comprehensive tests
        all_results = run_comprehensive_tests(epidata)
        
        # Analyze results
        summary_df = analyze_results(all_results)
        
        if summary_df is not None and not summary_df.empty:
            # Create visualizations
            create_visualizations(summary_df)
            
            # Print insights
            print_key_insights(summary_df)
        
        print("\n" + "="*60)
        print("MODEL TESTING COMPLETED!")
        print("="*60)
        print("Check the results in:")
        print("- data/graphs/model_test_results.csv")
        print("- data/graphs/model_test_results.png")
        
    except Exception as e:
        print(f"\nError in main execution: {e}")
        import traceback
        traceback.print_exc()



In [ ]:
main()

IMPROVED MODEL TESTING SCRIPT
This script tests improved GNN models with dense graphs.

COMPREHENSIVE MODEL TESTING

TESTING GRAPH TYPE: identity_graph
Validating graph structure for identity_graph...
logs saved
Graph stats: {'num_nodes': 411, 'num_edges': 411, 'density': 0.0024390243902439024, 'self_loops': 411, 'is_identity': True, 'is_connected': False, 'weight_stats': {'min': 1.0, 'max': 1.0, 'mean': 1.0, 'std': 0.0, 'non_zero': 411}}

Testing configuration: default

Testing improved_tgcn with identity_graph...
  Training improved_tgcn...
Dataloader Snapshot: Data(x=[411, 2, 8], edge_index=[2, 411], y=[411, 1], edge_weight=[411])
Epoch 1 train loss: 7.1751, val loss: 11.8032 ✓ (new best)
Epoch 2 train loss: 6.7715, val loss: 10.5971 ✓ (new best)
Epoch 3 train loss: 6.4770, val loss: 9.8923 ✓ (new best)
Epoch 4 train loss: 6.2661, val loss: 9.3053 ✓ (new best)
Epoch 5 train loss: 6.0560, val loss: 8.7110 ✓ (new best)
Epoch 6 train loss: 5.8214, val loss: 8.1149 ✓ (new best)
Epoch 7 